Name: Lin Qizhou

Key insights / takeaways:
- I used a vision-language workflow (Gemini) to extract a single scalar value (total_price) from receipt images with a strict, minimal-output prompt.
- I built a robust numeric parsing pipeline and stored raw model outputs for debugging, returning NaN when extraction failed.
- I evaluated performance on a random sample of receipts using MAE, extraction success rate, and threshold accuracy (within 5% / 10% of ground truth).

Results (test sample):
- Test receipts: [TOTAL_CNT]
- Successful extractions: [SUCCESS_CNT] ([SUCCESS_RATE])
- MAE (successful only): [MAE]
- Accuracy within 5%: [ACC_5]
- Accuracy within 10%: [ACC_10]


In [3]:
from google.colab import files

print("Upload ONE df_receipt file: CSV / Parquet / Pickle")
uploaded = files.upload()
print("Uploaded:", list(uploaded.keys()))


Upload ONE df_receipt file: CSV / Parquet / Pickle


Saving receipts_sample-20260102T154712Z-3-001 (1).zip to receipts_sample-20260102T154712Z-3-001 (1).zip
Saving receipt1 (1).jpg to receipt1 (1).jpg
Uploaded: ['receipts_sample-20260102T154712Z-3-001 (1).zip', 'receipt1 (1).jpg']


In [4]:
!pip -q install -U google-generativeai pandas tqdm pillow


     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 79.5/79.5 kB 2.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.9/10.9 MB 77.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 7.0/7.0 MB 84.1 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-colab 1.0.0 requires pandas==2.2.2, but you have pandas 3.0.1 which is incompatible.
bqplot 0.12.45 requires pandas<3.0.0,>=1.0.0, but you have pandas 3.0.1 which is incompatible.
db-dtypes 1.5.0 requires pandas<3.0.0,>=1.5.3, but you have pandas 3.0.1 which is incompatible.
gradio 5.50.0 requires pandas<3.0,>=1.0, but you have pandas 3.0.1 which is incompatible.
gradio 5.50.0 requires pillow<12.0,>=8.0, but you have pillow 12.1.1 which is incompatible.


In [9]:
from pathlib import Path


content_files = sorted([p.name for p in Path("/content").iterdir()])
print("✅ /content ：")
for name in content_files:
    print(" -", name)


ZIP_NAME = "receipts_sample-20260102T154712Z-3-001 (1).zip"
JPG_NAME = "receipt1 (1).jpg"

zip_path = Path("/content") / ZIP_NAME
jpg_path = Path("/content") / JPG_NAME

print("\n✅ ：")
print("zip exists? ", zip_path.exists(), "=>", zip_path)
print("jpg exists? ", jpg_path.exists(), "=>", jpg_path)


✅ /content ：
 - .config
 - receipt1 (1).jpg
 - receipts_sample-20260102T154712Z-3-001 (1).zip
 - sample_data
 - work

✅ ：
zip exists?  True => /content/receipts_sample-20260102T154712Z-3-001 (1).zip
jpg exists?  True => /content/receipt1 (1).jpg


In [10]:
from pathlib import Path
import zipfile

root = Path("/content")
zip_candidates = sorted(root.rglob("receipts_sample*.zip"))
if not zip_candidates:
    raise FileNotFoundError("receipts_sample*.zip not found under /content")

zip_path = zip_candidates[0]

workdir = Path("/content/work")
workdir.mkdir(parents=True, exist_ok=True)

with zipfile.ZipFile(zip_path, "r") as zf:
    zf.extractall(workdir)

receipts_dirs = sorted([p for p in workdir.rglob("receipts_sample") if p.is_dir()])
if not receipts_dirs:
    raise FileNotFoundError("receipts_sample directory not found after extraction")

RECEIPTS_DIR = receipts_dirs[0]

jpg_count = len(list(RECEIPTS_DIR.glob("*.jpg")))
txt_count = len(list(RECEIPTS_DIR.glob("*.txt")))

print(str(zip_path))
print(str(RECEIPTS_DIR))
print(jpg_count)
print(txt_count)


/content/receipts_sample-20260102T154712Z-3-001 (1).zip
/content/work/receipts_sample
714
716


In [11]:
import json
import re
import pandas as pd
from pathlib import Path

txt_files = sorted(RECEIPTS_DIR.glob("*.txt"))

money_pat = re.compile(r"[-+]?\d+(?:,\d{3})*(?:\.\d+)?")

def parse_total(txt_path: Path):
    raw = txt_path.read_text(encoding="utf-8", errors="ignore").strip()
    data = None
    try:
        inner = json.loads(raw)
        data = json.loads(inner) if isinstance(inner, str) else inner
    except Exception:
        try:
            data = json.loads(raw)
        except Exception:
            return None

    total_str = str((data or {}).get("total", "")).strip()
    m = money_pat.search(total_str.replace(" ", ""))
    if not m:
        return None
    try:
        return float(m.group(0).replace(",", ""))
    except Exception:
        return None

rows = []
for txt_path in txt_files:
    stem = txt_path.stem
    img_path = RECEIPTS_DIR / f"{stem}.jpg"
    if not img_path.exists():
        continue
    rows.append(
        {
            "receipt_id": stem,
            "image_path": str(img_path),
            "total_price": parse_total(txt_path),
        }
    )

df_receipt = pd.DataFrame(rows).dropna(subset=["total_price"]).reset_index(drop=True)
print(df_receipt.shape)
df_receipt.head()


(632, 3)


,receipt_id,image_path,total_price
0,receipt_sample_000,/content/work/receipts_sample/receipt_sample_0...,6.656
1,receipt_sample_001,/content/work/receipts_sample/receipt_sample_0...,85.060
2,receipt_sample_002,/content/work/receipts_sample/receipt_sample_0...,19.740
3,receipt_sample_003,/content/work/receipts_sample/receipt_sample_0...,47.520
4,receipt_sample_004,/content/work/receipts_sample/receipt_sample_0...,38000.000


In [12]:
N_TEST = 100
SEED = 42

if len(df_receipt) < N_TEST:
    raise ValueError(f"Not enough rows in df_receipt: {len(df_receipt)} < {N_TEST}")

df_test_receipts = df_receipt.sample(n=N_TEST, random_state=SEED).reset_index(drop=True)
print(df_test_receipts.shape)
df_test_receipts.head()


(100, 3)


,receipt_id,image_path,total_price
0,receipt_sample_374,/content/work/receipts_sample/receipt_sample_3...,18.75
1,receipt_sample_276,/content/work/receipts_sample/receipt_sample_2...,165.00
2,receipt_sample_645,/content/work/receipts_sample/receipt_sample_6...,147.60
3,receipt_sample_162,/content/work/receipts_sample/receipt_sample_1...,141.00
4,receipt_sample_558,/content/work/receipts_sample/receipt_sample_5...,46.54


In [18]:
!pip -q install -U requests


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 64.7/64.7 kB 2.1 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-colab 1.0.0 requires pandas==2.2.2, but you have pandas 3.0.1 which is incompatible.
google-colab 1.0.0 requires requests==2.32.4, but you have requests 2.32.5 which is incompatible.


In [23]:
from google.colab import userdata
import os

SECRET_NAME = "GEMINI_API_KEY"
os.environ["GOOGLE_API_KEY"] = userdata.get(SECRET_NAME)

print(os.environ.get("GOOGLE_API_KEY") is not None)


True


In [24]:
!pip -q install -U google-generativeai pillow tqdm

import os
import google.generativeai as genai

api_key = os.environ.get("GOOGLE_API_KEY")
if not api_key:
    raise RuntimeError("GOOGLE_API_KEY is not set")

genai.configure(api_key=api_key)
model = genai.GenerativeModel("gemini-1.5-flash")
print("ok")


ok


In [25]:
import re
from PIL import Image

prompt = "Extract the total amount from this receipt. Return ONLY the numerical value as a float (e.g., 24.55). No currency symbols, no extra text."

num_pat = re.compile(r"[-+]?\d+(?:,\d{3})*(?:\.\d+)?")

def parse_float(text):
    if not text:
        return None
    m = num_pat.search(text.strip().replace(" ", ""))
    if not m:
        return None
    try:
        return float(m.group(0).replace(",", ""))
    except Exception:
        return None

def gemini_predict_total(image_path):
    try:
        img = Image.open(image_path).convert("RGB")
        resp = model.generate_content([prompt, img])
        out = getattr(resp, "text", "") or ""
        return parse_float(out), out
    except Exception as e:
        return None, str(e)

test_path = df_test_receipts.loc[0, "image_path"]
pred, raw = gemini_predict_total(test_path)

print(test_path)
print(pred)
print(raw)


/content/work/receipts_sample/receipt_sample_374.jpg
None
unsupported image mode


In [26]:
from tqdm import tqdm
import numpy as np

preds = []
raws = []

for p in tqdm(df_test_receipts["image_path"].tolist()):
    pred, raw = gemini_predict_total(p)
    preds.append(pred)
    raws.append(raw)

df_test_receipts["predicted_total_price"] = preds
df_test_receipts["gemini_raw"] = raws

df_test_receipts[["receipt_id", "total_price", "predicted_total_price"]].head(15)


100%|██████████| 100/100 [00:01<00:00, 95.84it/s]


,receipt_id,total_price,predicted_total_price
0,receipt_sample_374,18.7500,None
1,receipt_sample_276,165.0000,None
2,receipt_sample_645,147.6000,None
3,receipt_sample_162,141.0000,None
4,receipt_sample_558,46.5400,None
5,receipt_sample_460,77.6000,None
6,receipt_sample_184,26.1100,None
7,receipt_sample_086,625.5000,None
8,receipt_sample_599,65.3965,None
9,receipt_sample_182,12.6112,None


In [27]:
import numpy as np

valid = df_test_receipts.dropna(subset=["total_price", "predicted_total_price"]).copy()
valid["abs_error"] = (valid["predicted_total_price"] - valid["total_price"]).abs()

total_cnt = len(df_test_receipts)
success_cnt = len(valid)
success_rate = success_cnt / total_cnt if total_cnt else 0.0
mae = float(valid["abs_error"].mean()) if success_cnt else np.nan

def threshold_accuracy(df, pct):
    if df.empty:
        return np.nan
    tol = df["total_price"].abs() * pct
    return float((df["abs_error"] <= tol).mean())

acc_5 = threshold_accuracy(valid, 0.05)
acc_10 = threshold_accuracy(valid, 0.10)

print(total_cnt)
print(success_cnt)
print(success_rate)
print(mae)
print(acc_5)
print(acc_10)

df_test_receipts[["receipt_id", "total_price", "predicted_total_price"]].head(15)


100
0
0.0
nan
nan
nan


,receipt_id,total_price,predicted_total_price
0,receipt_sample_374,18.7500,None
1,receipt_sample_276,165.0000,None
2,receipt_sample_645,147.6000,None
3,receipt_sample_162,141.0000,None
4,receipt_sample_558,46.5400,None
5,receipt_sample_460,77.6000,None
6,receipt_sample_184,26.1100,None
7,receipt_sample_086,625.5000,None
8,receipt_sample_599,65.3965,None
9,receipt_sample_182,12.6112,None


In [28]:
import pandas as pd
import numpy as np

valid = df_test_receipts.dropna(subset=["total_price", "predicted_total_price"]).copy()
valid["abs_error"] = (valid["predicted_total_price"] - valid["total_price"]).abs()

total_cnt = len(df_test_receipts)
success_cnt = len(valid)
success_rate = success_cnt / total_cnt if total_cnt else 0.0
mae = float(valid["abs_error"].mean()) if success_cnt else np.nan

def threshold_accuracy(df, pct):
    if df.empty:
        return np.nan
    tol = df["total_price"].abs() * pct
    return float((df["abs_error"] <= tol).mean())

acc_5 = threshold_accuracy(valid, 0.05)
acc_10 = threshold_accuracy(valid, 0.10)

summary = pd.DataFrame(
    [
        {"metric": "total_test_receipts", "value": total_cnt},
        {"metric": "successful_extractions", "value": success_cnt},
        {"metric": "success_rate", "value": success_rate},
        {"metric": "mae", "value": mae},
        {"metric": "accuracy_within_5pct", "value": acc_5},
        {"metric": "accuracy_within_10pct", "value": acc_10},
    ]
)

summary


,metric,value
0,total_test_receipts,100.0
1,successful_extractions,0.0
2,success_rate,0.0
3,mae,NaN
4,accuracy_within_5pct,NaN
5,accuracy_within_10pct,NaN


In [29]:
failed = df_test_receipts[df_test_receipts["predicted_total_price"].isna()].copy()
worst = valid.sort_values("abs_error", ascending=False).head(10)

print(len(failed))
failed[["receipt_id", "total_price", "predicted_total_price", "image_path"]].head(10)


100


,receipt_id,total_price,predicted_total_price,image_path
0,receipt_sample_374,18.7500,None,/content/work/receipts_sample/receipt_sample_3...
1,receipt_sample_276,165.0000,None,/content/work/receipts_sample/receipt_sample_2...
2,receipt_sample_645,147.6000,None,/content/work/receipts_sample/receipt_sample_6...
3,receipt_sample_162,141.0000,None,/content/work/receipts_sample/receipt_sample_1...
4,receipt_sample_558,46.5400,None,/content/work/receipts_sample/receipt_sample_5...
5,receipt_sample_460,77.6000,None,/content/work/receipts_sample/receipt_sample_4...
6,receipt_sample_184,26.1100,None,/content/work/receipts_sample/receipt_sample_1...
7,receipt_sample_086,625.5000,None,/content/work/receipts_sample/receipt_sample_0...
8,receipt_sample_599,65.3965,None,/content/work/receipts_sample/receipt_sample_5...
9,receipt_sample_182,12.6112,None,/content/work/receipts_sample/receipt_sample_1...


In [30]:
worst[["receipt_id", "total_price", "predicted_total_price", "abs_error", "image_path"]].head(10)


,receipt_id,total_price,predicted_total_price,abs_error,image_path


In [31]:
out_path = "/content/df_test_receipts_with_predictions.csv"
df_test_receipts.to_csv(out_path, index=False)
print(out_path)


/content/df_test_receipts_with_predictions.csv
